# PRODOVI · Horarios de publicación LSTM V4 · Facebook e Instagram

Entrenamiento híbrido por plataforma, condicionado al histórico de cada cuenta.
**El CSV inicial es SINTÉTICO: permite probar el proceso, no validar eficacia en Meta.**

1. En Colab selecciona Python 3 y, opcionalmente, GPU T4.
2. Ejecuta todas las celdas y sube `dataset_meta_v4.csv` cuando se solicite.
3. Revisa el reporte y descarga el ZIP al terminar.

El CSV inicial contiene 3.000 publicaciones simuladas: 1.500 Facebook y 1.500 Instagram,
con tres cuentas simuladas por plataforma y 500 publicaciones por cuenta.
Los patrones temporales fueron introducidos por un generador reproducible, no descubiertos en Meta.
Ambos modelos resultantes quedarán **experimentales** aunque sus métricas sean buenas.

El CSV original de 1.200 filas Facebook fue confirmado como ejemplo por el usuario y se conserva
convertido por separado en `dataset_facebook_legacy_v4.csv`. No se mezcló con el nuevo dataset.
Para entrenar con datos reales, sube un CSV del mismo contrato que conserve las cuentas y las
fechas reales de medición. Si falta una plataforma, quedará `no_data`.


In [ ]:
from pathlib import Path
import sys
import tensorflow as tf
import pandas as pd
import numpy as np
print('Python:', sys.version)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
WORK = Path('/content/prodovi_lstm_v4')
WORK.mkdir(parents=True, exist_ok=True)


## Contrato de datos y objetivo

Cada fila es una publicación de una cuenta identificada y una red: `facebook` o `instagram`.
`likes` significa reacciones en Facebook y me gusta en Instagram. Objetivo: **likes + 2 × comments**.
Es un puntaje ponderado, no una tasa ni una probabilidad. No es comparable numéricamente con el objetivo V3.

`published_at` y `metrics_observed_at` deben incluir offset, por ejemplo `2026-09-01T12:00:00-04:00`.
En datos Meta, la fecha de observación es obligatoria. La ventana solo usa métricas ya conocidas
antes de la publicación objetivo. No se mezclan cuentas, ni se completan faltantes con ceros.

En el dataset sintético se simula una medición a las 48 horas (`synthetic_fixed_48h`).
Para el legado se admite observación desconocida, con advertencia de evaluación exploratoria.
Un único snapshot actual de toda una cuenta no permite reconstruir qué métricas estaban disponibles
en el pasado. Puede terminar en `insufficient_causal_history`, aunque tenga muchas filas.
Recopila mediciones a una edad fija (por ejemplo 48 horas) para una evaluación temporal defendible.


In [ ]:
# Módulo autocontenido: data_pipeline.py
(WORK / 'data_pipeline.py').write_text('"""Contrato compartido de entrenamiento e inferencia. No consulta Meta ni inventa métricas."""\nimport numpy as np\nimport pandas as pd\n\nVERSION = \'meta_lstm_hybrid_v4.0.0\'\nTIMEZONE = \'America/La_Paz\'\nCOLUMNS = [\'platform\', \'account_id\', \'post_id\', \'published_at\', \'likes\', \'comments\',\n           \'metrics_observed_at\', \'source\', \'measurement_protocol\']\nHISTORY_FEATURES = [\'likes_log\', \'comments_log\', \'score_log\', \'hour_sin\',\n                    \'hour_cos\', \'day_sin\', \'day_cos\', \'gap_log\']\nCANDIDATE_FEATURES = [\'hour_sin\', \'hour_cos\', \'day_sin\', \'day_cos\', \'gap_log\',\n                      \'account_prior_log\', \'slot_samples_log\']\n\n\ndef read_dataset(path):\n    df = pd.read_csv(path, dtype=str, keep_default_na=False)\n    missing = set(COLUMNS) - set(df.columns)\n    if missing:\n        raise ValueError(f\'Faltan columnas: {sorted(missing)}\')\n    if df.empty:\n        raise ValueError(\'El CSV no contiene publicaciones.\')\n    for field in [\'platform\', \'account_id\', \'post_id\', \'published_at\', \'source\', \'measurement_protocol\']:\n        df[field] = df[field].str.strip()\n        if df[field].eq(\'\').any():\n            raise ValueError(f\'Hay valores vacíos en {field}.\')\n    if not df.platform.isin([\'facebook\', \'instagram\']).all():\n        raise ValueError(\'platform debe ser facebook o instagram.\')\n    if not df.source.isin([\'legacy_unverified\', \'meta_export\', \'synthetic\']).all():\n        raise ValueError(\'source debe ser legacy_unverified, meta_export o synthetic.\')\n    if df.duplicated([\'platform\', \'account_id\', \'post_id\']).any():\n        raise ValueError(\'Publicaciones duplicadas. Selecciona una medición por publicación.\')\n    for field in [\'likes\', \'comments\']:\n        df[field] = pd.to_numeric(df[field], errors=\'raise\')\n        if not (np.isfinite(df[field]) & df[field].ge(0) & df[field].mod(1).eq(0)).all():\n            raise ValueError(f\'{field} debe contener conteos enteros no negativos, nunca valores faltantes.\')\n    # No interpretar fechas sin offset como UTC accidentalmente.\n    for field in [\'published_at\', \'metrics_observed_at\']:\n        nonempty = df[field].ne(\'\')\n        if not df.loc[nonempty, field].str.contains(r\'(?:Z|[+-]\\d{2}:\\d{2})$\', regex=True).all():\n            raise ValueError(f\'{field} debe incluir zona horaria, por ejemplo -04:00.\')\n        df[field] = pd.to_datetime(df[field].replace(\'\', None), utc=True, errors=\'raise\', format=\'mixed\')\n    if (df.metrics_observed_at < df.published_at).any():\n        raise ValueError(\'Una medición no puede preceder a la publicación.\')\n    if (df.source.eq(\'meta_export\') & df.metrics_observed_at.isna()).any():\n        raise ValueError(\'Las exportaciones Meta necesitan metrics_observed_at.\')\n    df[\'score\'] = df.likes + 2 * df.comments\n    df[\'score_log\'] = np.log1p(df.score)\n    # Solo para legado exploratorio: no se conoce cuándo se midieron las métricas.\n    df[\'available_at\'] = df.metrics_observed_at.fillna(df.published_at)\n    return df.sort_values([\'published_at\', \'platform\', \'account_id\', \'post_id\']).reset_index(drop=True)\n\n\ndef calendar(timestamp):\n    local = pd.Timestamp(timestamp).tz_convert(TIMEZONE)\n    return [np.sin(2*np.pi*local.hour/24), np.cos(2*np.pi*local.hour/24),\n            np.sin(2*np.pi*local.dayofweek/7), np.cos(2*np.pi*local.dayofweek/7)]\n\n\ndef slot_prior(history, timestamp, strength=3.0):\n    """Referencia exclusiva de esta cuenta, calculada con datos disponibles antes del candidato."""\n    local = history.published_at.dt.tz_convert(TIMEZONE)\n    candidate = pd.Timestamp(timestamp).tz_convert(TIMEZONE)\n    slot = history[(local.dt.dayofweek == candidate.dayofweek) & (local.dt.hour == candidate.hour)]\n    average = float(history.score_log.mean())\n    prior = (float(slot.score_log.sum()) + strength * average) / (len(slot) + strength)\n    return prior, len(slot)\n\n\ndef make_inputs(history, timestamp, window, strength=3.0):\n    timestamp = pd.Timestamp(timestamp)\n    if timestamp.tzinfo is None:\n        raise ValueError(\'El candidato necesita zona horaria.\')\n    if history[[\'platform\', \'account_id\']].drop_duplicates().shape[0] != 1:\n        raise ValueError(\'La secuencia debe pertenecer a una sola cuenta y plataforma.\')\n    history = history[(history.published_at < timestamp) & (history.available_at < timestamp)]\n    history = history.sort_values([\'published_at\', \'post_id\'])\n    if len(history) < window:\n        raise ValueError(f\'Histórico insuficiente: requiere {window} publicaciones medidas antes del candidato.\')\n    prior, samples = slot_prior(history, timestamp, strength)\n    dates = history.published_at\n    gaps = dates.diff().dt.total_seconds().div(3600).clip(0, 720).fillna(24).to_numpy()\n    matrix = [[np.log1p(row.likes), np.log1p(row.comments), row.score_log,\n               *calendar(row.published_at), np.log1p(gap)]\n              for row, gap in zip(history.tail(window).itertuples(), gaps[-window:])]\n    gap = min(max((timestamp - dates.iloc[-1]).total_seconds()/3600, 0), 720)\n    candidate = [*calendar(timestamp), np.log1p(gap), prior, np.log1p(samples)]\n    return np.asarray(matrix[-window:], dtype=\'float32\'), np.asarray(candidate, dtype=\'float32\'), prior, samples\n\n\ndef build_examples(frame, window, strength=3.0):\n    histories, candidates, targets, records = [], [], [], []\n    for (platform, account), group in frame.groupby([\'platform\', \'account_id\'], sort=False):\n        for row in group.itertuples():\n            eligible = group[(group.published_at < row.published_at) & (group.available_at < row.published_at)]\n            if len(eligible) < window:\n                continue\n            h, c, prior, count = make_inputs(eligible, row.published_at, window, strength)\n            histories.append(h)\n            candidates.append(c)\n            targets.append(row.score_log - prior)\n            records.append(dict(platform=platform, account_id=account, post_id=row.post_id,\n                                published_at=row.published_at, available_at=row.available_at,\n                                actual=float(row.score), prior_log=prior, samples=count,\n                                global_log=float(eligible.score_log.mean())))\n    return (np.asarray(histories, dtype=\'float32\'), np.asarray(candidates, dtype=\'float32\'),\n            np.asarray(targets, dtype=\'float32\').reshape(-1, 1), pd.DataFrame(records))\n\n\ndef fit_scaler(values):\n    values = np.asarray(values)\n    return {\'mean\': values.mean(axis=0).tolist(), \'scale\': np.where(values.std(axis=0) < 1e-8, 1, values.std(axis=0)).tolist()}\n\n\ndef transform(values, scaler):\n    return ((np.asarray(values) - np.asarray(scaler[\'mean\'])) / np.asarray(scaler[\'scale\'])).astype(\'float32\')\n\n\ndef invert(values, scaler):\n    return np.asarray(values) * np.asarray(scaler[\'scale\']) + np.asarray(scaler[\'mean\'])\n\n\ndef combine(prior, residual, alpha):\n    # Evita overflow; el techo técnico no representa garantía de rendimiento.\n    return np.expm1(np.clip(np.asarray(prior) + alpha*np.asarray(residual), 0, 20))\n', encoding='utf-8')


In [ ]:
# Módulo autocontenido: train.py
(WORK / 'train.py').write_text('"""Entrena en Colab; un modelo por red, ventanas exclusivas por cuenta, prueba temporal."""\nimport hashlib\nimport json\nimport platform as runtime_platform\nimport shutil\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\n\nfrom data_pipeline import (VERSION, TIMEZONE, HISTORY_FEATURES, CANDIDATE_FEATURES,\n                           read_dataset, build_examples, fit_scaler, transform, invert, combine)\n\nSEED = 623\nWINDOWS = [3, 7, 14]\nSTRENGTH = 3.0\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef metrics(actual, predicted):\n    actual, predicted = np.asarray(actual), np.asarray(predicted)\n    error = actual - predicted\n    total = float(np.sum((actual - actual.mean())**2))\n    correlation = pd.Series(actual).rank().corr(pd.Series(predicted).rank())\n    return dict(MAE=float(np.abs(error).mean()), RMSE=float(np.sqrt((error**2).mean())),\n                R2=float(1 - np.sum(error**2)/total) if total else None,\n                Spearman=float(correlation) if np.isfinite(correlation) else None)\n\n\ndef model_for(window):\n    history = tf.keras.layers.Input((window, len(HISTORY_FEATURES)), name=\'history_sequence\')\n    candidate = tf.keras.layers.Input((len(CANDIDATE_FEATURES),), name=\'candidate_slot\')\n    h = tf.keras.layers.LSTM(32, dropout=0.15)(history)\n    c = tf.keras.layers.Dense(12, activation=\'relu\')(candidate)\n    x = tf.keras.layers.Concatenate()([h, c])\n    x = tf.keras.layers.Dense(24, activation=\'relu\')(x)\n    x = tf.keras.layers.Dropout(0.15)(x)\n    output = tf.keras.layers.Dense(1, name=\'residual_log\')(x)\n    model = tf.keras.Model([history, candidate], output)\n    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=\'mse\')\n    return model\n\n\ndef train_platform(frame, network, output, epochs=100):\n    # Fronteras globales por red: ninguna cuenta entrena con fechas futuras de otra.\n    dates = sorted(frame.published_at.unique())\n    if len(frame) < 150 or len(dates) < 30:\n        return dict(status=\'insufficient_data\', rows=len(frame),\n                    reason=\'Se requieren al menos 150 filas y 30 fechas/horas distintas por red para este piloto.\')\n    train_end, val_end = dates[int(len(dates)*0.70)], dates[int(len(dates)*0.85)]\n    out = output / network\n    out.mkdir()\n    selected, experiments = None, []\n    for window in WINDOWS:\n        h, c, y, records = build_examples(frame, window, STRENGTH)\n        if records.empty:\n            experiments.append(dict(window=window, status=\'no_causal_sequences\'))\n            continue\n        masks = {\n            \'train\': ((records.published_at < train_end) & (records.available_at < train_end)).to_numpy(),\n            \'validation\': ((records.published_at >= train_end) & (records.published_at < val_end)\n                           & (records.available_at < val_end)).to_numpy(),\n            \'test\': (records.published_at >= val_end).to_numpy(),\n        }\n        counts = {name: int(mask.sum()) for name, mask in masks.items()}\n        if counts[\'train\'] < 60 or counts[\'validation\'] < 12 or counts[\'test\'] < 12:\n            experiments.append(dict(window=window, status=\'insufficient_sequences\', **counts))\n            continue\n        tf.keras.backend.clear_session()\n        tf.keras.utils.set_random_seed(SEED)\n        scalers = dict(history=fit_scaler(h[masks[\'train\']].reshape(-1, h.shape[-1])),\n                       candidate=fit_scaler(c[masks[\'train\']]), residual=fit_scaler(y[masks[\'train\']]))\n        hs, cs, ys = transform(h, scalers[\'history\']), transform(c, scalers[\'candidate\']), transform(y, scalers[\'residual\'])\n        model = model_for(window)\n        inputs = lambda mask: {\'history_sequence\': hs[mask], \'candidate_slot\': cs[mask]}\n        print(f\'{network}: ventana {window}, secuencias {counts}\', flush=True)\n        history = model.fit(inputs(masks[\'train\']), ys[masks[\'train\']],\n                            validation_data=(inputs(masks[\'validation\']), ys[masks[\'validation\']]),\n                            epochs=epochs, batch_size=32, shuffle=False, verbose=2,\n                            callbacks=[tf.keras.callbacks.EarlyStopping(monitor=\'val_loss\', patience=12, restore_best_weights=True),\n                                       tf.keras.callbacks.ReduceLROnPlateau(monitor=\'val_loss\', patience=5, factor=0.5, min_lr=1e-5)])\n        residual = invert(model.predict(inputs(masks[\'validation\']), verbose=0), scalers[\'residual\']).ravel()\n        val = records.loc[masks[\'validation\']]\n        scores = [(float(alpha), metrics(val.actual, combine(val.prior_log, residual, alpha))[\'MAE\'])\n                  for alpha in np.linspace(0, 1, 21)]\n        alpha, mae = min(scores, key=lambda item: item[1])\n        experiments.append(dict(window=window, status=\'trained\', alpha=alpha, validation_MAE=mae,\n                                best_epoch=int(np.argmin(history.history[\'val_loss\'])+1), **counts))\n        if selected is None or mae < selected[\'mae\']:\n            # Guardar ahora: clear_session del siguiente experimento no altera el artefacto elegido.\n            model.save(out / \'model.keras\')\n            selected = dict(mae=mae, window=window, alpha=alpha, scalers=scalers,\n                            records=records, masks=masks, hs=hs, cs=cs,\n                            history=history.history, counts=counts)\n    save_json(out / \'validation_selection.json\', experiments)\n    if selected is None:\n        return dict(status=\'insufficient_causal_history\', rows=len(frame),\n                    reason=\'No hay suficientes métricas conocidas antes de cada candidato y frontera temporal.\',\n                    experiments=experiments)\n    s = selected\n    model = tf.keras.models.load_model(out / \'model.keras\', compile=False)\n    mask = s[\'masks\'][\'test\']\n    test = s[\'records\'].loc[mask].copy()\n    test_inputs = {\'history_sequence\': s[\'hs\'][mask], \'candidate_slot\': s[\'cs\'][mask]}\n    residual = invert(model.predict(test_inputs, verbose=0), s[\'scalers\'][\'residual\']).ravel()\n    test[\'prediction_lstm_hybrid\'] = combine(test.prior_log, residual, s[\'alpha\'])\n    test[\'prediction_slot_history\'] = combine(test.prior_log, 0, 0)\n    test[\'prediction_account_mean_log\'] = combine(test.global_log, 0, 0)\n    report = {name: metrics(test.actual, test[column]) for name, column in {\n        \'lstm_hybrid\': \'prediction_lstm_hybrid\', \'slot_history\': \'prediction_slot_history\',\n        \'account_mean_log\': \'prediction_account_mean_log\'}.items()}\n    baseline = min(report[\'slot_history\'][\'MAE\'], report[\'account_mean_log\'][\'MAE\'])\n    improvement = 100*(baseline-report[\'lstm_hybrid\'][\'MAE\'])/baseline if baseline else None\n    per_account = []\n    for account, group in test.groupby(\'account_id\'):\n        per_account.append(dict(account_id=account, samples=len(group),\n                                hybrid=metrics(group.actual, group.prediction_lstm_hybrid),\n                                historical=metrics(group.actual, group.prediction_slot_history)))\n    # Incluso un buen resultado retrospectivo necesita verificación de procedencia y un piloto prospectivo.\n    verified = bool(frame.source.eq(\'meta_export\').all())\n    protocols = sorted(frame.measurement_protocol.unique().tolist())\n    controlled = bool(len(protocols) == 1 and protocols[0].startswith(\'fixed_\'))\n    offline_better = bool(s[\'alpha\'] > 0 and baseline > report[\'lstm_hybrid\'][\'MAE\'])\n    metadata = dict(model_version=VERSION, platform=network, window=s[\'window\'], alpha=s[\'alpha\'],\n                    smoothing_strength=STRENGTH, timezone=TIMEZONE, day_convention=\'0=Monday,6=Sunday\',\n                    history_features=HISTORY_FEATURES, candidate_features=CANDIDATE_FEATURES,\n                    engagement_formula=\'likes + 2 * comments; Facebook likes means reactions\',\n                    output_unit=\'weighted_interaction_score_not_percentage\',\n                    inference_contract=\'data_pipeline.make_inputs -> scalers -> model -> combine\',\n                    personalized_by=\'same-account causal history and same-account slot prior; no account embedding\',\n                    dataset_rows=len(frame), accounts=int(frame.account_id.nunique()),\n                    sources=sorted(frame.source.unique().tolist()), measurement_protocols=protocols,\n                    verified_provenance=verified, controlled_measurement=controlled,\n                    offline_lstm_beats_baselines=offline_better, test_improvement_percent=improvement,\n                    ready_for_production=False, requires_prospective_validation=True,\n                    status=\'candidate_for_review\' if verified and controlled and offline_better else \'experimental_only\',\n                    split={\'strategy\': \'global_chronological_per_platform_with_label_availability\',\n                           \'train_before\': str(train_end), \'validation_before\': str(val_end), **s[\'counts\']},\n                    limitations=[\'No demuestra que cambiar la hora cause más interacciones.\',\n                                 \'No incluye contenido, campañas pagadas ni tamaño de audiencia.\',\n                                 \'Legado sin fecha de medición usa disponibilidad aproximada: evaluación exploratoria.\',\n                                 \'Un identificador legacy no acredita pertenencia a una cuenta real.\',\n                                 \'MAE/RMSE evalúan publicaciones observadas; no validan horas sin publicaciones.\',\n                                 \'No activar Instagram con un modelo entrenado únicamente para Facebook.\'])\n    save_json(out / \'metadata.json\', metadata)\n    save_json(out / \'scalers.json\', s[\'scalers\'])\n    save_json(out / \'test_metrics.json\', report)\n    save_json(out / \'per_account_metrics.json\', per_account)\n    test.to_csv(out / \'test_predictions.csv\', index=False)\n    pd.DataFrame(s[\'history\']).to_csv(out / \'training_history.csv\', index=False)\n    np.savez_compressed(out / \'inference_smoke_test.npz\', history=s[\'hs\'][mask][:2],\n                        candidate=s[\'cs\'][mask][:2],\n                        expected_residual_scaled=model.predict({k: v[:2] for k, v in test_inputs.items()}, verbose=0))\n    print(network, metadata[\'status\'], report, flush=True)\n    return dict(status=metadata[\'status\'], window=s[\'window\'], alpha=s[\'alpha\'], metrics=report,\n                test_improvement_percent=improvement, ready_for_production=False)\n\n\ndef run_training(csv_path, output_parent=\'artifacts\', epochs=100):\n    tf.keras.utils.set_random_seed(SEED)\n    try:\n        tf.config.experimental.enable_op_determinism()\n    except (AttributeError, RuntimeError):\n        pass\n    frame = read_dataset(csv_path)\n    # Directorio nuevo por ejecución: un ZIP nunca hereda un modelo de otra ejecución.\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    output = Path(output_parent) / f\'lstm_horarios_meta_v4_{stamp}\'\n    output.mkdir(parents=True, exist_ok=False)\n    report = {}\n    for network in (\'facebook\', \'instagram\'):\n        subset = frame[frame.platform.eq(network)].copy()\n        report[network] = train_platform(subset, network, output, epochs) if len(subset) else {\'status\': \'no_data\'}\n    save_json(output / \'training_report.json\', report)\n    save_json(output / \'environment.json\', dict(python=runtime_platform.python_version(), tensorflow=tf.__version__,\n              keras=tf.keras.__version__, numpy=np.__version__, pandas=pd.__version__, seed=SEED,\n              dataset_sha256=hashlib.sha256(Path(csv_path).read_bytes()).hexdigest()))\n    for name in (\'data_pipeline.py\', \'predict.py\', \'train.py\'):\n        shutil.copyfile(Path(__file__).parent / name, output / name)\n    save_json(output / \'deployment_gate.json\', dict(ready_for_production=False,\n              instruction=\'Revisar procedencia, métricas, cobertura por cuenta y piloto antes de integrar recomendaciones.\'))\n    archive = shutil.make_archive(str(output), \'zip\', root_dir=output)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n    print(\'ZIP para devolver al proyecto:\', archive)\n    return archive, report\n', encoding='utf-8')


In [ ]:
# Módulo autocontenido: predict.py
(WORK / 'predict.py').write_text('"""Referencia de inferencia para la futura integración. No publica contenido."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\n\nfrom data_pipeline import make_inputs, transform, invert, combine, TIMEZONE, VERSION\n\n\ndef rank_slots(model_dir, account_history, candidates):\n    folder = Path(model_dir)\n    meta = json.loads((folder / \'metadata.json\').read_text(encoding=\'utf-8\'))\n    scalers = json.loads((folder / \'scalers.json\').read_text(encoding=\'utf-8\'))\n    if meta[\'model_version\'] != VERSION:\n        raise ValueError(\'Versión de contrato incompatible.\')\n    if account_history.empty or account_history[[\'platform\', \'account_id\']].drop_duplicates().shape[0] != 1:\n        raise ValueError(\'Entrega el histórico de una sola cuenta y red.\')\n    if account_history.platform.iloc[0] != meta[\'platform\']:\n        raise ValueError(\'No se puede usar el modelo de una red para otra.\')\n    model = tf.keras.models.load_model(folder / \'model.keras\', compile=False)\n    hs, cs, priors, records = [], [], [], []\n    now = pd.Timestamp.now(tz=TIMEZONE)\n    # Congelar información conocida ahora: no simular métricas futuras entre candidatos.\n    available = account_history[(account_history.available_at <= now) & (account_history.published_at <= now)]\n    for timestamp in candidates:\n        timestamp = pd.Timestamp(timestamp)\n        if timestamp.tzinfo is None or timestamp <= now:\n            raise ValueError(\'Todos los candidatos deben ser futuros y tener zona horaria.\')\n        h, c, prior, count = make_inputs(available, timestamp, meta[\'window\'], meta[\'smoothing_strength\'])\n        hs.append(h)\n        cs.append(c)\n        priors.append(prior)\n        records.append(dict(timestamp=timestamp.tz_convert(TIMEZONE).isoformat(), samples_in_slot=count,\n                            account_id=str(available.account_id.iloc[0]), platform=meta[\'platform\'],\n                            historical_score=float(np.expm1(prior)), status=meta[\'status\'],\n                            ready_for_production=False, unseen_slot=count == 0))\n    if not records:\n        return []\n    raw = model.predict({\'history_sequence\': transform(hs, scalers[\'history\']),\n                         \'candidate_slot\': transform(cs, scalers[\'candidate\'])}, verbose=0)\n    residual = invert(raw, scalers[\'residual\']).ravel()\n    predicted = combine(priors, residual, meta[\'alpha\'])\n    for row, value in zip(records, predicted):\n        row[\'predicted_score\'] = float(value)\n    return sorted(records, key=lambda row: row[\'predicted_score\'], reverse=True)\n', encoding='utf-8')


In [ ]:
# Recargar módulos si vuelves a ejecutar el notebook en la misma sesión.
sys.path.insert(0, str(WORK)) if str(WORK) not in sys.path else None
for name in ('train', 'predict', 'data_pipeline'):
    sys.modules.pop(name, None)
from google.colab import files
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_names) != 1:
    raise ValueError('Sube exactamente un CSV unificado. No subas el CSV original V3.')
CSV_PATH = WORK / 'dataset_meta_v4.csv'
CSV_PATH.write_bytes(uploaded[csv_names[0]])
from data_pipeline import read_dataset
dataset = read_dataset(CSV_PATH)
display(dataset.groupby(['platform', 'source']).agg(publicaciones=('post_id', 'size'), cuentas=('account_id', 'nunique')))
print('Protocolos de medición:', dataset.measurement_protocol.unique().tolist())
print('Con observación conocida:', int(dataset.metrics_observed_at.notna().sum()), '/', len(dataset))
if not dataset.source.eq('meta_export').all():
    print('AVISO: hay datos de procedencia no verificada o sintéticos. Resultado experimental.')
if 'instagram' not in dataset.platform.values:
    print('Instagram: sin datos. No se fabricará un modelo para esa red.')


## Entrenamiento y evaluación

Se prueban ventanas de 3, 7 y 14 publicaciones. La selección de ventana y peso LSTM usa validación.
El 70 % / 15 % / 15 % se delimita cronológicamente por plataforma; se excluyen etiquetas que no
estaban disponibles antes de la frontera del conjunto. Los normalizadores se ajustan solo con entrenamiento.
Los promedios históricos de cada ejemplo se calculan solo con observaciones anteriores de su cuenta.

Se requieren 150 filas por red y al menos 60 / 12 / 12 secuencias de entrenamiento / validación / prueba.
Son umbrales operativos para el experimento, no una garantía de suficiencia estadística.

La prueba es secuencial: un resultado de prueba ya observado puede formar parte del histórico de
una predicción posterior. Los pesos del modelo no se actualizan. Esto reproduce inferencias sucesivas.
Si el peso LSTM elegido es cero, la validación prefirió el método histórico; no se fuerza una mejora.


In [ ]:
from train import run_training
EPOCHS = 100  # Usa 2 solo para comprobar ejecución; para el experimento conserva 100.
archive, report = run_training(CSV_PATH, WORK / 'artifacts', epochs=EPOCHS)


In [ ]:
import json
import matplotlib.pyplot as plt
artifact_dir = Path(archive).with_suffix('')
for network in ('facebook', 'instagram'):
    print(network.upper(), json.dumps(report[network], ensure_ascii=False, indent=2))
    folder = artifact_dir / network
    if not (folder / 'test_predictions.csv').exists():
        continue
    results = pd.read_csv(folder / 'test_predictions.csv')
    history = pd.read_csv(folder / 'training_history.csv')
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    axes[0].plot(history.loss, label='Entrenamiento')
    axes[0].plot(history.val_loss, label='Validación')
    axes[0].set_title(f'{network}: pérdida del residuo normalizado')
    axes[0].legend()
    ordered = results.sort_values('published_at')
    for column, label in [('actual', 'Real'), ('prediction_lstm_hybrid', 'LSTM híbrida'), ('prediction_slot_history', 'Histórico')]:
        axes[1].plot(ordered[column].to_numpy(), label=label, alpha=0.8)
    axes[1].set_title(f'{network}: prueba temporal · puntaje ponderado')
    axes[1].legend()
    plt.show()
    display(pd.DataFrame(json.loads((folder / 'per_account_metrics.json').read_text())))


## Comprobar artefacto y descargar

El ZIP contiene modelos `.keras`, normalizadores JSON, contrato de características, métricas globales
y por cuenta, predicciones de prueba, versiones de bibliotecas y código de inferencia.
La prueba siguiente verifica que el modelo guardado reproduce su salida, no su eficacia comercial.

**Ningún modelo se declara listo para producción automáticamente.** La mejora de MAE no prueba
una mejora causal por cambiar la hora, ni una buena clasificación de horarios nunca observados.
La validación prospectiva y la integración Laravel se harán después de revisar este ZIP.


In [ ]:
for network in ('facebook', 'instagram'):
    folder = artifact_dir / network
    if not (folder / 'inference_smoke_test.npz').exists():
        continue
    fixture = np.load(folder / 'inference_smoke_test.npz', allow_pickle=False)
    restored = tf.keras.models.load_model(folder / 'model.keras', compile=False)
    actual = restored.predict({'history_sequence': fixture['history'], 'candidate_slot': fixture['candidate']}, verbose=0)
    np.testing.assert_allclose(actual, fixture['expected_residual_scaled'], rtol=1e-5, atol=1e-6)
    print(network, 'modelo exportado verificado')
files.download(archive)


## Referencias técnicas

- [TensorFlow: series temporales, ventanas y separación temporal](https://www.tensorflow.org/tutorials/structured_data/time_series).
- [Keras: guardado y recuperación de modelos](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/).
